# 03 — Treinamento Interativo da Rede (`AffinePNN` × `DNN_AN`)

Use este notebook quando quiser **treinar diretamente via Jupyter/SWAN** em vez de submeter job ao CERN HTCondor (`condor/train_both.sub`).

Todo o código da arquitetura, amostragem por quotas e otimizador **é importado diretamente dos módulos Python em `src/` e dos YAMLs em `configs/`** — garantindo que um modelo treinado aqui ou no HTCondor seja 100% idêntico e salve exatamente os mesmos checkpoints em `outputs/checkpoints/`.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import tensorflow as tf

from src.data.preprocessing import check_four_vectors, make_inputs, prepare_datasets_from_config
from src.data.sampler import pool_report, verify_generator_quotas
from src.data.weights import make_eval_weights, oversampling_frequency_weights, weight_report
from src.evaluation.plotting import plot_training_history
from src.models.affine_dnn import build_model_from_config
from src.training.train import create_generator_for_model, save_model_bundle, train_model
from src.utils.config import load_config
from src.utils.paths import OutputPaths
from src.utils.seed import configure_gpu_memory_growth, set_all_seeds

print("TensorFlow:", tf.__version__, "| GPUs:", configure_gpu_memory_growth())

## 1. Carregamento da Configuração (`configs/`)

Você pode ajustar hiperparâmetros diretamente em `configs/training.yaml`, `configs/affine.yaml` e `configs/dnn_an.yaml`, ou passar *overrides* pontuais na lista `OVERRIDES` abaixo.

In [ ]:
USE_SYNTHETIC_IF_MISSING = False
MODELS_TO_TRAIN = ["affine", "dnn_an"]   # Opções em configs/: "affine", "dnn_an", "pnn_concat"

OVERRIDES = [
    # Descomente para um teste curto de sanidade antes do treino completo de 40 épocas:
    # "training.epochs=3",
    # "training.steps_per_epoch=200",
]

cfg = load_config(PROJECT_ROOT / "configs" / "default.yaml", overrides=OVERRIDES)
set_all_seeds(cfg["seed"])
out_paths = OutputPaths.from_root(cfg["paths"]["output_dir"], create=True)

if not Path(cfg["paths"]["parquet_dir"]).exists() and USE_SYNTHETIC_IF_MISSING:
    from src.data.synthetic import generate_synthetic_parquets
    cfg["paths"]["parquet_dir"] = str(generate_synthetic_parquets(PROJECT_ROOT / "data" / "synthetic", cfg, seed=cfg["seed"]))

print("Resumo de treino carregado do YAML:")
print(json.dumps(cfg["training"], indent=2))
print("\nArquitetura AffinePNN:")
print(json.dumps(cfg["models"]["affine"], indent=2))

## 2. Preparação dos Dados e Salva de Normalização / Encoders

Salva `normalization_train_only.csv` e `config_encoders.json` em `outputs/checkpoints/` para uso posterior nos notebooks 04 e 05.

In [ ]:
data = prepare_datasets_from_config(
    cfg,
    save_checkpoints_dir=out_paths.checkpoints,
    verbose=True,
)

check_four_vectors(
    x_cont=data.x_cont["train"],
    x_cat=data.x_cat["train"],
    scaler=data.scaler,
    continuous_features=cfg["features"]["continuous_features"],
    lbn_particles=cfg["features"]["lbn_particles"],
    mask_bjet1_without_pair=bool(cfg["features"].get("mask_bjet1_always_on_no_pair", False)),
    strict=True,
    verbose=True,
)

## 3. Gerador de Mini-Batches com Quotas (AN-25-103 Seção 5.3.5) e Pesos de Monitoramento (Seção 5.3.6)

- Verifica empiricamente (em 300 batches) que a fração sorteada de cada subprocesso reproduz sua seção de choque líquida (`verify_generator_quotas`).
- Diagnostica a taxa de repetição por evento em pools pequenos (`pool_report`).
- Calcula `val_freq_w` para o *early stopping* dos dois modelos.

In [ ]:
gen_affine = create_generator_for_model(cfg, "affine", data)
bx, by = gen_affine[0]
print("Formato de um mini-batch:", {k: v.shape for k, v in bx.items()})
print("Composição por classe no batch:", {cfg["physics"]["class_labels"][c]: int((by == c).sum()) for c in cfg["physics"]["class_labels"]})

quota_check = verify_generator_quotas(gen_affine, data.splits["train"], n_batches=200, verbose=True)
pools_df = pool_report(gen_affine, steps=len(gen_affine), epochs=cfg["training"]["epochs"], verbose=True)

val_df = data.splits["val"]
val_eval_w = make_eval_weights(val_df, cfg["physics"]["class_labels"])
weight_report(val_df, val_eval_w, cfg["physics"]["class_labels"], title="VALIDAÇÃO: pesos físicos balanceados (val_eval_w — métricas pós-treino)")

val_freq_w = oversampling_frequency_weights(
    val_df,
    gen_affine,
    class_fraction=cfg["sampling"]["class_batch_fraction"],
    clip=cfg["sampling"]["val_freq_clip"],
)
weight_report(val_df, val_freq_w, cfg["physics"]["class_labels"], title="VALIDAÇÃO: frequência do oversampling (val_freq_w — early stopping)")

## 4. Inspeção das Arquiteturas (`src/models/affine_dnn.py`) antes de Treinar

In [ ]:
for m_key in MODELS_TO_TRAIN:
    m_preview = build_model_from_config(cfg, m_key, data.scaler, data.cat_preprocessor.vocab_sizes)
    m_name = cfg["models"][m_key]["name"]
    print(f"{m_name:12s} (kind={cfg['models'][m_key]['kind']:6s}) -> {m_preview.count_params():,d} parâmetros")
    if cfg["models"][m_key]["kind"] == "none":
        assert not any("affine" in l.name for l in m_preview.layers), "DNN_AN não deve conter camadas afins!"
    del m_preview

## 5. Loop de Treinamento e Persistência em `outputs/checkpoints/`

In [ ]:
val_inputs = make_inputs(
    data.x_cont["val"],
    data.x_cat["val"],
    val_df["kappa_eval"].to_numpy(),
    kappa_norm=cfg["physics"]["kappa_norm"],
)
val_y = val_df["label"].to_numpy()

trained_models = {}
histories = {}
metas = {}

for m_key in MODELS_TO_TRAIN:
    m_name = cfg["models"][m_key]["name"]
    gen = create_generator_for_model(cfg, m_key, data)
    vw = oversampling_frequency_weights(
        val_df,
        gen,
        class_fraction=cfg["sampling"]["class_batch_fraction"],
        clip=cfg["sampling"]["val_freq_clip"],
    )
    model, hist_dict, meta = train_model(
        cfg=cfg,
        model_key=m_key,
        generator=gen,
        scaler=data.scaler,
        cat_vocab_sizes=data.cat_preprocessor.vocab_sizes,
        val_data=(val_inputs, val_y, vw),
        name=m_name,
    )
    paths_saved = save_model_bundle(out_paths, m_name, model, hist_dict, meta)
    print(f"Salvo {m_name} em: {paths_saved['model']}")

    trained_models[m_name] = model
    histories[m_name] = hist_dict
    metas[m_name] = meta

with open(out_paths.checkpoints / "run_config_snapshot.json", "w", encoding="utf-8") as f:
    json.dump({"config": cfg, "models_trained": metas}, f, indent=2)

## 6. Exportação Completa dos Modelos (`ONNX`, `ONNX End-to-End`, `SavedModel`, `.keras`, `.weights.h5`, `.npz`)

Salva em `outputs/exported_models/<Modelo>/` todos os formatos portáveis de inferência (incluindo o grafo **ONNX End-to-End** que embute todo o pré-processamento físico e padronização) e valida a paridade numérica no `onnxruntime`.

In [ ]:
from src.models.export import export_complete_model_bundle

export_manifests = {}
for m_name, model in trained_models.items():
    manifest = export_complete_model_bundle(
        model_name=m_name,
        model=model,
        scaler=data.scaler,
        cat_preprocessor=data.cat_preprocessor,
        cfg=cfg,
        export_root_dir=out_paths.exported_models,
        history_dict=histories[m_name],
        meta_dict=metas[m_name],
        opset=int(cfg.get("export", {}).get("onnx_opset", 17)),
        verify_parity=True,
    )
    export_manifests[m_name] = manifest
    parity = manifest["onnx_parity_verification"]
    print(
        f"[OK] Pacote {m_name} salvo em {manifest['bundle_dir']} | "
        f"ONNX max|ΔP| (pre={parity['preprocessed_onnx_max_abs_diff']:.2e}, "
        f"end2end={parity['end2end_raw_onnx_max_abs_diff']:.2e})"
    )

## 7. Resumo Rápido da Convergência

In [ ]:
display(pd.DataFrame(metas).T[["kind", "backbone", "n_params", "epochs_ran", "best_epoch", "best_val_loss", "elapsed_seconds"]])

plot_training_history(
    histories,
    save_path=out_paths.plots / "training_history.png",
    show=True,
)